In [1]:
from pathlib import Path

dir_evals = Path('docs/temp/evals')
if not dir_evals.exists(): print(f"Directory {dir_evals} does not exist.")

tools = ['chatgpt_deepresearch', 'manus_ai', 'discourse2draft']

In [6]:
import re
from src.backend.ai.llms import getAIModel
from pydantic import BaseModel, Field
from langchain.output_parsers.fix import OutputFixingParser
from langchain_core.output_parsers import PydanticOutputParser
from src.backend.ai.prompts import setPrompt
from src.backend.utils import Config
import pandas as pd

class StandardizeSectionHeadersSchema(BaseModel):
    '''
    Returns the mapping from the section header to standard section header
    '''
    mapping: dict[str, str] = Field(description='A mapping from the section header to standard section header')

def standardizeHeader(section_headers, section_headers_standard):

    llm = getAIModel(model_name='azure-gpt-4o')
    system_prompt = 'You are an AI assist designed to standardize section headers in a document'
    human_prompt = '''
                You will be provided with a document which is a list of section headers.
                You will also be provided with a list of standard section headers.
                Your task is to map each provided section header to a standard section header.
                If there is no mapping possible, map the section header to itself.

                <Section headers>
                {section_headers}
                </Section headers>

                <Standard section headers>
                {section_headers_standard}
                </Standard section headers>
                '''
    
    parser = OutputFixingParser.from_llm(parser=PydanticOutputParser(pydantic_object=StandardizeSectionHeadersSchema), 
                                llm=llm,
                                max_retries=Config.RETRY_COUNTER)
    
    prompt = setPrompt(system_prompt, human_prompt, parser)
        
    response = (prompt | llm | parser).invoke(input={'section_headers': section_headers, 'section_headers_standard': section_headers_standard})
    return dict(response)['mapping']

def extractSectionContent(file_path):
    
    with open(file_path) as fp:
        data = '\n'.join([line.strip() for line in fp.readlines()])
    
    output = re.findall(r'(#+ .+\n?)([^#]+)', data)
    return [(section.strip(), content.strip()) for section, content in output]

def getSectionAndContent(file_path, section_headers_standard):
    doc = extractSectionContent(file_path)
    doc_san = []
    for section, content in doc:
        if section.startswith('# '):
            doc_san.append(['# Title', section[2:]])
        elif section.startswith('## '):
            doc_san.append([section, content])
        else:
            doc_san[-1][1] += section + '\n' + content
    section_headers = [section for section, _ in doc_san]
    mapping = standardizeHeader(section_headers, section_headers_standard)
    return [(mapping.get(section, section), content) for section, content in doc_san]
        
def getSectionsForComparison(file_name):

    section_headers_standard = [item for item in extractSectionContent(dir_evals / 'prompts' / file_name) if not item[0].startswith('# ')]
    responses = {tool: dict(getSectionAndContent(dir_evals / tool / file_name, section_headers_standard)) for tool in tools}
    
    return pd.DataFrame(responses)


In [9]:
file_names = ['CRISPR-based editing for inherited blood disorders.md', 'Phthalates Toxicity.md', 'PFAS.md']

for file_name in file_names:
    print(f'Processing "{file_name}"...')
    df_sections = getSectionsForComparison(file_name)
    df_sections.to_csv(dir_evals / 'sections_to_compare' / f'{'.'.join(file_name.split('.')[:-1])}.csv')
    display(df_sections)

Processing "CRISPR-based editing for inherited blood disorders.md"...


Calling src.backend.ai.llms.getAIModel

2026-01-28 09:40:37 [INFO] Calling src.backend.ai.llms.getAIModel


Calling src.backend.ai.prompts.setPrompt

2026-01-28 09:40:37 [INFO] Calling src.backend.ai.prompts.setPrompt
2026-01-28 09:40:42 [INFO] HTTP Request: POST https://litellm.toxpipe.niehs.nih.gov/chat/completions "HTTP/1.1 200 OK"
2026-01-28 09:40:44 [INFO] HTTP Request: POST https://litellm.toxpipe.niehs.nih.gov/chat/completions "HTTP/1.1 200 OK"


...

Calling src.backend.ai.llms.getAIModel

2026-01-28 09:41:14 [INFO] Calling src.backend.ai.llms.getAIModel


Calling src.backend.ai.prompts.setPrompt

2026-01-28 09:41:14 [INFO] Calling src.backend.ai.prompts.setPrompt
2026-01-28 09:41:15 [INFO] HTTP Request: POST https://litellm.toxpipe.niehs.nih.gov/chat/completions "HTTP/1.1 200 OK"
2026-01-28 09:41:17 [INFO] HTTP Request: POST https://litellm.toxpipe.niehs.nih.gov/chat/completions "HTTP/1.1 200 OK"


,chatgpt_deepresearch,manus_ai,discourse2draft
# Title,Health Impacts of Per- and Polyfluoroalkyl Sub...,Health Impacts of Per- and Polyfluoroalkyl Sub...,Title: Health Impacts of Per- and Polyfluoroal...
## Abstract:,Per- and polyfluoroalkyl substances (PFASs) ar...,Per- and polyfluoroalkyl substances (PFASs) re...,Per- and polyfluoroalkyl substances (PFAS) con...
## Introduction & Chemical Background:,Per- and polyfluoroalkyl substances (PFASs) re...,Per- and polyfluoroalkyl substances (PFASs) ar...,Per- and polyfluoroalkyl substances (PFAS) are...
## Exposure Pathways & Toxicokinetics:,**Exposure Pathways:** Human exposure to PFASs...,### 2.1. Exposure Pathways\nHuman exposure to ...,Drinking water contamination has emerged as a ...
## System-Specific Health Impacts (The Core Analysis):,### Immunotoxicity\nOne of the most sensitive ...,The evaluation of health impacts is based on t...,PFAS exposure has been consistently linked to ...
## Mechanisms of Action:,Elucidating how PFASs cause the array of obser...,The biological plausibility of the observed he...,PFAS interactions with peroxisome proliferator...
"## The ""GenX"" Problem:","By the early 2000s, scientific and regulatory ...",The phase-out of PFOA and PFOS led to the intr...,GenX (perfluoro-2-methyl-3-oxahexanoic acid) w...
## Regulatory Context & Knowledge Gaps:,**Regulatory Context:** Confronted with mounti...,### 6.1. Regulatory Context\nRegulatory bodies...,In its recent proposal under the Safe Drinking...
## Conclusion:,PFASs have transitioned from scientific obscur...,The weight of evidence from both epidemiologic...,Mounting epidemiological data underscore that ...
## References,1. **American Bar Association (ABA)**. (2022)....,[1]: https://pmc.ncbi.nlm.nih.gov/articles/PMC...,"[1] Sukhram, S. D., Kim, J., Musovic, S., Anid..."


In [10]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain.output_parsers.fix import OutputFixingParser
from pydantic import BaseModel, Field
from src.backend.ai.prompts import setPrompt
from src.backend.ai.common import StateContentManager
from src.backend.utils import Config

class ScoreWithReasonSchema(BaseModel):
    '''
    Returns a score within 0 to 100 for a particular criterion and a statement supporting the score
    '''
    score: int = Field(description='Score within 0 to 100')
    reason: str = Field(description='A short statement supporting the score')

class RateContentSchema(BaseModel):
    '''
    Returns scores based on different criteria to rate the content of a structured document
    '''

    relevance: ScoreWithReasonSchema = Field(description='A score with supporting statement that evaluates the Relevance of the content')
    continuity: ScoreWithReasonSchema = Field(description='A score with supporting statement that evaluates the Continuity of the content')
    non_repetitiveness: ScoreWithReasonSchema = Field(description='A score with supporting statement that evaluates the Uniqueness (or non-repetitiveness) of the content')
    specificity: ScoreWithReasonSchema = Field(description='A score with supporting statement that evaluates the Specificity of the content')
    #relevance of citation (hallucination), accuracy of the content+citation (hallucination), specificity of the content

# ---------------------------------------------------------------------------
class RateContent:

    rate_content_system_prompt = '''\
    You will be provided a manuscript section header with or without previously written content summary on a specific topic. You are a scholarly reviewer with expertise in the corresponding topic area. Your task is review and rate the section content.
    
    <Instructions>
    - Rating must be an integer number within 0 (lowest) and 100 (highest).
    - Rating will base on the following criteria.
    **Relevance:** Is the content relevant to the provided section header and the summary (if provided).
    **Continuity:** Is the content writing style is continuous with the previously written content summary. Put the highest score if previously written content summary is not provided.
    **Non-Repetitiveness:** Is the content devoid of repetitiveness compared with the previously written content summary. Put the highest score if previously written content summary is not provided.
    **Specificity:** Is the content devoid of halucination.
    </Instructions>
    '''

    rate_content_human_prompt = '''
    <Previous Content Summary>
    {content_pre}
    </Previous Content Summary>

    <Current Section Header>
    {current_section}
    </Current Section Header>

    <Content>
    {content}
    </Content>
    
    <Instructions>
    - Read the Previous Content Summary. 
    - Review the provided content based on the Current Section Header
    - Provide the output in the following format.
    {format_instructions}
    
    - Output must be in JSON format with `json` tags.
    </Instructions>
    '''

    def __init__(self, llm):

        parser = OutputFixingParser.from_llm(parser=PydanticOutputParser(pydantic_object=RateContentSchema), 
                                             llm=llm,
                                             max_retries=Config.RETRY_COUNTER)
        
        self.rate_content_prompt = setPrompt(self.rate_content_system_prompt, self.rate_content_human_prompt, parser)
        
        self.rate_content_chain = self.rate_content_prompt | llm | parser


    def __call__(self, state: StateContentManager):
        '''LLM generates reports from a given outline'''
        
        response = self.rate_content_chain.invoke(input={'content_pre': state['content_pre'],
                                                             'current_section': state['current_section'],
                                                             'content': state['content']})
        try:
            content = dict(response)
        except:
            raise Exception(f'GenerateContent response does not have content, response: {response}')

        return {'content': content, 'steps': ['Rate Content']}
    
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, END, StateGraph
from src.backend.ai.llms import getAIModel
from src.backend.ai.summarize import Summarize
from typing import Literal

# -----------------------------------------------------------------------
def check_if_summary_needed(
        state: StateContentManager,
    ) -> Literal['Summarize', 'Rate Content']:
        if len(state.get('content_pre').split()) > 500:
            return 'Summarize'
        return 'Rate Content'

# -----------------------------------------------------------------------
class ContentWriterArchitecture:
     
     def __init__(self, model_name, temperature):
        llm = getAIModel(model_name=model_name, temperature=temperature)

        # Define a new graph
        workflow = StateGraph(state_schema=StateContentManager)

        # Define the (single) node in the graph
        workflow.add_node("Summarize", Summarize(llm=llm, input_field='content_pre'))
        workflow.add_node("Rate Content", RateContent(llm=llm))
        #workflow.add_node("Add Citations", AddCitations(llm))

        workflow.add_conditional_edges(START, check_if_summary_needed)
        workflow.add_edge("Summarize", "Rate Content")

        self.agent = workflow.compile()


In [11]:
agent = ContentWriterArchitecture(model_name='azure-o3', temperature=0).agent

Calling src.backend.ai.llms.getAIModel

2026-01-28 09:44:00 [INFO] Calling src.backend.ai.llms.getAIModel


Calling src.backend.ai.prompts.setPrompt

2026-01-28 09:44:00 [INFO] Calling src.backend.ai.prompts.setPrompt


Calling src.backend.ai.prompts.setPrompt

2026-01-28 09:44:00 [INFO] Calling src.backend.ai.prompts.setPrompt


In [14]:
import tqdm
import pandas as pd

# relevance, continuity, uniqueness, relevance of citation (hallucination), accuracy of the content+citation (hallucination), specificity of the content

from pathlib import Path

def formatRating(rating, tool):
    rating_dict = {}
    for criterion in rating:
        rating_dict[f'{tool} {criterion} (score)'] = rating[criterion].score
        rating_dict[f'{tool} {criterion} (reason)'] = rating[criterion].reason

    return rating_dict

file_names = ['CRISPR-based editing for inherited blood disorders.md', 'Phthalates Toxicity.md', 'PFAS.md']

for file_name in file_names:
    print(f'Processing "{file_name}"...')
    section_sets = pd.read_csv(dir_evals / 'sections_to_compare' / f'{'.'.join(file_name.split('.')[:-1])}.csv', index_col=0)
    content_pre_sets = ['' for _ in section_sets]
    rating_responses = {}
    for index, row in tqdm.tqdm(section_sets.iterrows()):
        
        section_header = row.name
        print(section_header)
        
        rating_respons_section_wise = {}
        for i, tool in enumerate(tools):
            
            print(tool)

            content = row[tool] if not pd.isna(row[tool]) else ''

            if content.strip() == '': continue
            
            rating_response = agent.invoke({'current_section': section_header, 'content': content, 'content_pre': content_pre_sets[i]})['content']
            rating_respons_section_wise |= formatRating(rating_response, tool)

            content_pre_sets[i] += f'{section_header}\n\n{content}'

        rating_responses[section_header] = rating_respons_section_wise

    rating_responses = pd.DataFrame(rating_responses)

    rating_score = rating_responses.loc[rating_responses.index.str.endswith('(score)')]
    rating_reason = rating_responses.loc[rating_responses.index.str.endswith('(reason)')]

    rating_score = rating_score.rename(index=lambda x:x.replace(' (score)', ''))
    rating_reason = rating_reason.rename(index=lambda x:x.replace(' (reason)', ''))

    rating_score.to_csv(dir_evals / 'scores' / f'{Path(file_name).stem}.csv', index=True)
    rating_reason.to_csv(dir_evals / 'reasons' / f'{Path(file_name).stem}.csv', index=True)

Processing "CRISPR-based editing for inherited blood disorders.md"...


0it [00:00, ?it/s]

# Title
chatgpt_deepresearch


2026-01-28 09:48:25 [INFO] HTTP Request: POST https://litellm.toxpipe.niehs.nih.gov/chat/completions "HTTP/1.1 200 OK"


manus_ai


...

Calling src.backend.ai.common.extractLLMResponse

2026-01-28 10:10:24 [INFO] Calling src.backend.ai.common.extractLLMResponse
2026-01-28 10:10:34 [INFO] HTTP Request: POST https://litellm.toxpipe.niehs.nih.gov/chat/completions "HTTP/1.1 200 OK"
2026-01-28 10:10:42 [INFO] HTTP Request: POST https://litellm.toxpipe.niehs.nih.gov/chat/completions "HTTP/1.1 200 OK"


discourse2draft


Calling src.backend.ai.common.extractLLMResponse

2026-01-28 10:10:42 [INFO] Calling src.backend.ai.common.extractLLMResponse
2026-01-28 10:10:57 [INFO] HTTP Request: POST https://litellm.toxpipe.niehs.nih.gov/chat/completions "HTTP/1.1 200 OK"
2026-01-28 10:11:02 [INFO] HTTP Request: POST https://litellm.toxpipe.niehs.nih.gov/chat/completions "HTTP/1.1 200 OK"
10it [07:25, 44.53s/it]


In [15]:
rating_score.rename(index=lambda x:x.replace(' (score)', ''))

,# Title,## Abstract:,## Introduction & Chemical Background:,## Exposure Pathways & Toxicokinetics:,## System-Specific Health Impacts (The Core Analysis):,## Mechanisms of Action:,"## The ""GenX"" Problem:",## Regulatory Context & Knowledge Gaps:,## Conclusion:,## References
chatgpt_deepresearch relevance,95,95,95,97,95,95,96,95,95,95
chatgpt_deepresearch continuity,100,95,90,92,90,90,92,90,90,90
chatgpt_deepresearch non_repetitiveness,100,100,75,88,70,85,94,88,80,96
chatgpt_deepresearch specificity,90,90,90,93,85,88,88,85,90,85
manus_ai relevance,95,95,90,95,95,95,95,92,95,90
manus_ai continuity,98,94,85,90,90,90,90,87,90,85
manus_ai non_repetitiveness,100,96,90,96,80,85,85,90,80,40
manus_ai specificity,90,88,88,88,88,90,88,84,92,60
discourse2draft relevance,90,92,95,95,95,95,90,95,90,95
discourse2draft continuity,95,98,90,90,90,90,85,90,85,90


In [16]:
rating_responses = pd.DataFrame(rating_responses)

rating_responses.loc[rating_responses.index.str.endswith('(reason)')]

,# Title,## Abstract:,## Introduction & Chemical Background:,## Exposure Pathways & Toxicokinetics:,## System-Specific Health Impacts (The Core Analysis):,## Mechanisms of Action:,"## The ""GenX"" Problem:",## Regulatory Context & Knowledge Gaps:,## Conclusion:,## References
chatgpt_deepresearch relevance (reason),The section header is # Title; the supplied co...,The content is clearly an abstract that succin...,The section squarely addresses chemical struct...,Content directly addresses exposure pathways a...,The section thoroughly addresses system-specif...,Discussion directly addresses the header 'Mech...,The section focuses squarely on GenX—its origi...,Content focuses exactly on regulatory developm...,The section appropriately serves as a conclusi...,Section header calls for references; list prov...
chatgpt_deepresearch continuity (reason),"No prior content summary exists, so continuity...",Only a title was provided previously; the writ...,"Tone, terminology, and scope align well with t...","Tone, depth, and structure are consistent with...","Writing style, depth, and framing align well w...",Builds naturally on the previous summary (whic...,"Tone, terminology, and level of detail align w...","Tone, depth, and structure align with the scho...",Tone and subject matter align well with the pr...,Transition from narrative summary to annotated...
chatgpt_deepresearch non_repetitiveness (reason),With no previous text to repeat and only a sin...,"With no prior body text to duplicate, the abst...",Some inevitable overlap with the abstract (e.g...,Only limited overlap with the summary (brief m...,While new mechanistic details and study exampl...,"While some mechanisms (PPAR activation, immune...",Only minimal overlap with the previous summary...,While it reiterates that PFAS are regulated an...,"Some repetition of facts (health endpoints, re...",References do not duplicate prior narrative te...
chatgpt_deepresearch specificity (reason),"The title is specific, accurate, and free from...","Information is accurate and well-focused, citi...","Chemical descriptions, historical dates, produ...","Provides quantitative half-lives, biological m...","Information is generally accurate, nuanced, an...","Cites well-established molecular targets, spec...","Provides concrete chemical name, timeline, geo...","Provides concrete regulatory values, dates, an...","Statements about health outcomes, regulatory l...",Most citations are concrete and plausible; som...
manus_ai relevance (reason),"The section header is ""# Title,"" and the conte...",The text functions as a concise scientific abs...,"Content directly introduces PFAS chemistry, st...",Content squarely addresses the declared sectio...,"The subsection delivers detailed, endpoint-spe...",The text directly describes biological mechani...,Directly focuses on GenX and other short-chain...,Material directly addresses both halves of the...,Content clearly functions as a succinct conclu...,A list of citations is exactly what is expecte...
manus_ai continuity (reason),No prior summary exists; the provided title se...,"With only a title provided previously, the wri...",Builds logically from the abstract by expandin...,"Writing style, terminology, and citation forma...",Content smoothly elaborates on themes (immunot...,"Builds seamlessly on the prior summary, expand...",Flows logically from the prior summary’s menti...,"Tone, terminology and focus align well with th...","Tone, terminology, and focus align well with t...","The cited papers cover exposure pathways, immu..."
manus_ai non_repetitiveness (reason),"With no previous content, there is no repetiti...","No earlier narrative exists to duplicate, and ...",Adds new foundational details (C–F bond streng...,Material introduces new information (exposure ...,Although it revisits endpoints mentioned earli...,"Some overlap with the earlier summary (PPAR-α,...","Adds new points (EPA toxicity comparison, wate...","Adds fresh information (specific MCL values